# Stock Market Trend Classification: Decision Tree vs LSTM

Classifying Apple (AAPL) daily stock movement as **bullish (1)** or **bearish (0)** using 18 price & technical-indicator features (2014–2023).

Sections: data loading → EDA (7 charts) → preprocessing → Decision Tree → LSTM → comparison.

In [2]:

# !pip install pandas numpy matplotlib seaborn scikit-learn tensorflow

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout




In [3]:
# Load the dataset (place aapl_2014_2023.csv in the data/ folder — see data/README.md)
# Running in Google Colab instead? Uncomment the two lines below to upload the file:
# from google.colab import files
# uploaded = files.upload()

df = pd.read_csv('../data/aapl_2014_2023.csv')
df['date'] = pd.to_datetime(df['date'])  # needed for the yearly grouping later

df.head()


         date       open       high        low      close     volume  \
0  2014-01-02  19.845715  19.893929  19.715000  19.754642  234684800   
1  2014-01-03  19.745001  19.775000  19.301071  19.320715  392467600   
2  2014-01-06  19.194643  19.528570  19.057142  19.426071  412610800   
3  2014-01-07  19.440001  19.498571  19.211430  19.287144  317209200   
4  2014-01-08  19.243214  19.484285  19.238930  19.409286  258529600   

       rsi_7     rsi_14       cci_7      cci_14     sma_50     ema_50  \
0  44.149614  50.996175 -105.071637  -42.411877  19.282236  19.265060   
1  33.094332  43.883529 -152.150999 -124.511983  19.297314  19.267243   
2  37.525661  45.858019 -132.306490 -130.122350  19.310864  19.273472   
3  34.055624  43.675708  -89.587009 -108.089019  19.316671  19.274008   
4  39.768497  46.104107  -58.525539  -82.150032  19.329171  19.279313   

     sma_100    ema_100      macd  bollinger  TrueRange     atr_7    atr_14  \
0  18.372604  18.484713  0.194637  19.979268   0.

In [4]:
# Dataset shape
print("Dataset shape:", df.shape)

Dataset shape: (2516, 20)


In [5]:
# Dataset columns
print("\nColumns available:")
print(df.columns)



Columns available:
Index(['date', 'open', 'high', 'low', 'close', 'volume', 'rsi_7', 'rsi_14',
       'cci_7', 'cci_14', 'sma_50', 'ema_50', 'sma_100', 'ema_100', 'macd',
       'bollinger', 'TrueRange', 'atr_7', 'atr_14', 'target'],
      dtype='object')


In [6]:
# Checking missing values
print("\nMissing values per column:")
print(df.isnull().sum())


Missing values per column:
date         0
open         0
high         0
low          0
close        0
volume       0
rsi_7        0
rsi_14       0
cci_7        0
cci_14       0
sma_50       0
ema_50       0
sma_100      0
ema_100      0
macd         0
bollinger    0
TrueRange    0
atr_7        0
atr_14       0
target       0
dtype: int64


**Visualization**

In [12]:
#Closing Price Over Time
plt.figure(figsize=(14,6))
plt.plot(df['date'], df['close'], color='blue')
plt.title('Apple Closing Price Over Time (2014–2023)')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.grid(True)
plt.show()



[Chart saved in ../images/ — see the project README]

**Graph 1:**  
The long term uptrend of Appl’s closing stock price between 2014 and 2023 is strong.  
Recoveries are clearly visible from the significant dips, be it from the 2020 pandemic case.  
This represents an uptrend which is indicative of how far Apple is growing and being resilient in the market.



In [13]:
#Distribution of Daily True Range (Volatility)
plt.figure(figsize=(10,5))
plt.hist(df['TrueRange'], bins=50, color='purple', edgecolor='black')
plt.title('Distribution of Daily True Range (Volatility)')
plt.xlabel('True Range')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 2:**  
The True Range was low for most days in a row, showing that daily volatility of the market was low.  
True Range values were very high for a few days, which meant that there had been significant price moves.  
Being aware of volatility is helpful for traders to make sense of the risks and determine what levels to set stop loss at.


In [14]:
#Yearly Average Closing Price
df['Year'] = df['date'].dt.year
yearly_avg = df.groupby('Year')['close'].mean()

plt.figure(figsize=(12,6))
yearly_avg.plot(kind='bar', color='orange')
plt.title('Yearly Average Closing Price')
plt.xlabel('Year')
plt.ylabel('Average Closing Price (USD)')
plt.grid(True)
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 3:**  
The data seen above has a steady growth that is evident on Appl’s yearly average closing price which especially increased after the year 2020.  
After a sharp rise from 2020, the recovery is very strong and investor sentiment is very bullish.  
Tracking average yearly averages allows the measurement of long term investments performance.



In [15]:
#Volume vs Close Price
plt.figure(figsize=(10,6))
plt.scatter(df['volume'], df['close'], alpha=0.5, color='green')
plt.title('Volume vs Closing Price')
plt.xlabel('Volume')
plt.ylabel('Closing Price (USD)')
plt.grid(True)
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 4:**  
A higher closings price is normally most common for lower trading volumes, and spikes of high volume occur seldom.  
In these data, there is little or no strong direct link between volume and price.  
The scatter plots are used to visualize possible correlations amount trading activity and stock price.



In [16]:
#RSI 7 Distribution
plt.figure(figsize=(8,5))
sns.boxplot(x=df['rsi_7'], color='cyan')
plt.title('Distribution of RSI 7')
plt.xlabel('RSI 7')
plt.grid(True)
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 5:**  
Most of the RSI 7 values are between 40 and 70, which usually point to slightly bullish momentum.  
On the lower end there are a few outliers, and on the higher end as well.  
Boxplots can be used as a guide to visualize the spread of the data, the central tendency, and also to detect extreme values.


In [19]:
#ATR 14 (Volatility)

plt.figure(figsize=(10,5))
sns.kdeplot(df['atr_14'], fill=True, color='red')

plt.title('Density Plot of ATR 14 (Volatility)')
plt.xlabel('ATR 14')
plt.ylabel('Density')
plt.grid(True)
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 6:**  
The ATR 14 value mostly ranges between 0 and 2, which indicates low average volatility.  
But they occur less commonly and point to the time when the market gyrates the most.  
They offer a smooth view of how volatility is spread through time.



In [18]:
corr_cols = ['open', 'high', 'low', 'close', 'volume',
             'rsi_7', 'rsi_14', 'cci_7', 'cci_14', 'macd', 'TrueRange']

corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(14,10))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Selected Features')
plt.show()


[Chart saved in ../images/ — see the project README]

**Graph 7:**  
Stock data is strongly correlated for Open, High, Low, and Close prices as expected.  
The price features have weak negative correlation with volume, and RSI and CCI indicators are highly related.  
This helps figure out strong feature relationships on the predictive modeling.


**Model preparing and training**

In [20]:
#Step 1: Select Features and Target
# Import necessary libraries
import pandas as pd
from sklearn.model_selection import train_test_split

# Select features and target
feature_cols = ['open', 'high', 'low', 'close', 'volume',
                'rsi_7', 'rsi_14', 'cci_7', 'cci_14',
                'sma_50', 'ema_50', 'sma_100', 'ema_100',
                'macd', 'bollinger', 'TrueRange', 'atr_7', 'atr_14']

X = df[feature_cols]
y = df['target']

print("Features Shape:", X.shape)
print("Target Shape:", y.shape)


Features Shape: (2516, 18)
Target Shape: (2516,)


In [51]:
# Remove rows where target == 'neutral'
neutral_indices = y[y == 'neutral'].index

# Drop those from X and y
X = X.drop(index=neutral_indices)
y = y.drop(index=neutral_indices)

# Check result
print("Unique values in y after cleaning:", y.unique())

Unique values in y after cleaning: [0 1]


In [52]:
# Two versions of y
y_dt = y.copy()  # for Decision Tree (keep as integers)
y_lstm = np.array(y).astype(np.float32)  # for LSTM (convert to float32)


In [53]:
#Step 2: Train-Test Split
from sklearn.model_selection import train_test_split

# 80% train, 20% test — NO shuffling
X_train, X_test, y_train_dt, y_test_dt = train_test_split(X, y_dt, test_size=0.2, shuffle=False, random_state=42)
_, _, y_train_lstm, y_test_lstm = train_test_split(X, y_lstm, test_size=0.2, shuffle=False, random_state=42)

print(X_train.shape, X_test.shape)
print(y_train_dt.shape, y_test_dt.shape)


(1384, 18) (346, 18)
(1384,) (346,)


In [54]:
#Step 3: Scaling Data (for LSTM)
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Initialize Scaler
scaler = MinMaxScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for LSTM (samples, time_steps=1, features)
X_train_lstm = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_lstm = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

print(X_train_lstm.shape, X_test_lstm.shape)

(1384, 1, 18) (346, 1, 18)


In [56]:
# Force y_train_dt and y_test_dt to integer type
y_train_dt = y_train_dt.astype(int)
y_test_dt = y_test_dt.astype(int)

print(y_train_dt.dtype, y_test_dt.dtype)


int64 int64


In [62]:
#Step 4: Decision Tree Model (Training and Evaluation)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report

# Train Decision Tree
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train_dt)

# Predict
y_pred_dt = dt_model.predict(X_test)

# Evaluate
print("Decision Tree Accuracy:", accuracy_score(y_test_dt, y_pred_dt))
print("Decision Tree Precision:", precision_score(y_test_dt, y_pred_dt))


Decision Tree Accuracy: 0.5375722543352601
Decision Tree Precision: 0.5531914893617021


In [61]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Already have these from your previous code
y_pred_dt = dt_model.predict(X_test)
cm = confusion_matrix(y_test_dt, y_pred_dt)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Decision Tree Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


[Chart saved in ../images/ — see the project README]

In [63]:

print("\nClassification Report:\n", classification_report(y_test_dt, y_pred_dt))


Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.49      0.51       166
           1       0.55      0.58      0.57       180

    accuracy                           0.54       346
   macro avg       0.54      0.54      0.54       346
weighted avg       0.54      0.54      0.54       346



In [58]:
#STEP 5: Build and Train LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense

# Build Model
lstm_model = Sequential()
lstm_model.add(Input(shape=(X_train_lstm.shape[1], X_train_lstm.shape[2])))
lstm_model.add(LSTM(50, return_sequences=False))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Train
history = lstm_model.fit(X_train_lstm, y_train_lstm, epochs=20, batch_size=32, validation_split=0.1, verbose=1)


Epoch 1/20
39/39  5s 33ms/step - accuracy: 0.4671 - loss: 0.6934 - val_accuracy: 0.5252 - val_loss: 0.6993
Epoch 2/20
39/39  0s 8ms/step - accuracy: 0.5698 - loss: 0.6853 - val_accuracy: 0.5252 - val_loss: 0.7002
...
Epoch 20/20
39/39  1s 9ms/step - accuracy: 0.5670 - loss: 0.6815 - val_accuracy: 0.5324 - val_loss: 0.6921


In [65]:
#Step 6: Evaluate LSTM
# Predict
y_pred_lstm = (lstm_model.predict(X_test_lstm) > 0.5).astype(int)

# Evaluate
print("LSTM Accuracy:", accuracy_score(y_test_lstm, y_pred_lstm))
print("LSTM Precision:", precision_score(y_test_lstm, y_pred_lstm))



11/11  0s 7ms/step 
LSTM Accuracy: 0.5
LSTM Precision: 0.5132075471698113


In [64]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix


# Create confusion matrix
cm_lstm = confusion_matrix(y_test_lstm, y_pred_lstm)

# Plot confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Greens', cbar=True)
plt.title('LSTM Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


[Chart saved in ../images/ — see the project README]

In [66]:

print("\nClassification Report:\n", classification_report(y_test_lstm, y_pred_lstm))


Classification Report:
               precision    recall  f1-score   support

         0.0       0.46      0.22      0.30       166
         1.0       0.51      0.76      0.61       180

    accuracy                           0.50       346
   macro avg       0.48      0.49      0.46       346
weighted avg       0.49      0.50      0.46       346



In [60]:
#Step 7 comparison
import pandas as pd

# Create comparison DataFrame
comparison = pd.DataFrame({
    'Model': ['Decision Tree', 'LSTM'],
    'Accuracy': [accuracy_score(y_test_dt, y_pred_dt), accuracy_score(y_test_lstm, y_pred_lstm)],
    'Precision': [precision_score(y_test_dt, y_pred_dt), precision_score(y_test_lstm, y_pred_lstm)]
})

print(comparison)


           Model  Accuracy  Precision
0  Decision Tree  0.537572   0.553191
1           LSTM  0.500000   0.513208
